# 04.1 — Frontend, backend, and HTTP

[Class index and setup](README.md) · [Next: 04.2](04.2-a-small-validated-client.ipynb)

**Purpose:** Place a request in the application architecture before writing HTTP code.

**In class:** 30 minutes. **Independent work:** 15 minutes, included in the index's 210-minute allowance. Preparation has its own 30 minutes. The instructor's separate project is additional work.

## Learning outcomes

- Distinguish the frontend, backend, API contract, and data source.
- Identify the method, address, path, parameters, status, and body.
- Inspect a supplied FastAPI route and change one query parameter.

## Starting point

You know basic Python, functions, JSON-like dictionaries, and the earlier Observation/Report idea. The complete package and fixtures are supplied again; no previous notebook or exercise answer is needed. Open the supplied class workspace in VS Code, select this class's `.venv`, then **Restart Kernel → Run All**.

Default cells inspect saved responses and run offline checks. They do not start a server, call an external API, read credentials, install software, or download models. Live commands are explicit **manual terminal steps**. An HTTP-shaped fixture is not evidence of a live request. Empty answer cells are safe.

Attempt exercises before using the separate `%load` cell: uncomment its last line and run to reveal, then run the populated cell separately. A loaded solution replaces its loader cell; restarting the kernel does not restore that source. Keep your attempt in its own cell.

In [ ]:
from pathlib import Path
import sys

if not Path("report_app/api.py").is_file() or not Path("fixtures/observations.json").is_file():
    raise FileNotFoundError(
        "Open 04-consuming-apis.code-workspace and restart the kernel. "
        "Download the complete class folder, including report_app/, fixtures/, and solutions/."
    )
print("Notebook kernel:", sys.executable)
print("Class assets available; default execution is offline.")

### Visual guide — Identify roles before machines

<img src="images/client-server.png" alt="A client sends a request to a server, which returns a response." width="850" style="max-width: 100%; height: auto; background-color: white;">

Requests is our client; FastAPI is the supplied server. They can run on the same laptop. The client initiates a request and receives a response; it need not be a browser. Keep the server terminal running while using the client terminal.

*Credit: [Client-server model](https://commons.wikimedia.org/wiki/File:Client-server_model.svg) by Lubaochuan, [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). Commons PNG rendering; no content changes. [Sources and teaching notes](IMAGE_SUGGESTIONS.md).*

## In class · 0–10 minutes — Who talks to whom?

A **frontend** is the part a person interacts with: a browser page, a form, or a notebook acting as a user interface. A **backend** is the program handling requests and applying rules or reading data. An **API** is the interface a program exposes to other programs. Our backend happens to expose an HTTP API. These are roles, not separate programming languages.

```text
Person → browser/notebook → HTTP request → FastAPI backend → saved observations
                         ← status + JSON ←                ←
```

In this class the notebook is a client. The supplied FastAPI program is a server. Both can run on one laptop. `127.0.0.1` means this machine; `8000` identifies the listening service. An API is not inherently an internet service. Later, the public-data client contacts a different server; a backend can itself be another API's client.

The existing `Report` and summary functions stay ordinary Python. The server supplies data; the client validates it; the calculation uses the validated records. Moving a request into a class constructor would make merely creating an object depend on a server.

### A request and a response

| Part | Our example | Meaning |
| --- | --- | --- |
| Method | `GET` | Ask to read a resource |
| Base address | `http://127.0.0.1:8000` | Where this server listens |
| Path | `/observations` | Which resource to request |
| Query | `?label=north` | Ask for one exact label |
| Status | `200` | HTTP success |
| Response header | `Content-Type: application/json` | Describes the body format |
| Body | `[{"label": "north", "value": 2.5}]` | The returned data |

JSON is text on the wire. `response.json()` turns it into Python lists/dictionaries. A successful status does not guarantee the body has the fields our program expects.

Optional illustration: [client/server diagram, creator, license, and placement notes](IMAGE_SUGGESTIONS.md). Trace the request and reply; both roles can run on one laptop.

In [ ]:
import requests
from examples.saved_http import saved_response
from report_app.api import LOCAL_URL

request = requests.Request("GET", LOCAL_URL, params={"label": "north"}).prepare()
response = saved_response("north.json")
print("Prepared only; not sent:", request.method, request.url)
print("Saved response:", response.status_code, response.headers["Content-Type"])
print("JSON text:", response.text)
print("Python object:", response.json())

## In class · 10–20 minutes — The supplied FastAPI backend

Read the short route below. `FastAPI()` creates the application. The decorator associates `GET /observations` with a Python function. Because `label` is not in the path, it is a query parameter. With no label the function returns all records; with a label it returns exact matches. An unknown label produces a successful empty list, not a server error.

The reusable code lives in [service.py](report_app/service.py). We inspect it here rather than maintain a second copy. Students consume this supplied service; writing a new backend is optional.

In [ ]:
import inspect
from report_app.service import observations

print(inspect.getsource(observations))
print("Direct Python call, not HTTP:", observations(label="north"))

### Manual terminal demonstration — included in this teaching block

In terminal A, from this class directory:

```sh
uv run --locked python -m uvicorn report_app.service:app --host 127.0.0.1 --port 8000
```

Leave it running. In a browser open `http://127.0.0.1:8000/observations?label=north`. Open `/docs` to inspect the interactive API interface and try its parameter. The docs page is a browser frontend; FastAPI still handles its requests. The standard docs UI loads external assets, so if they are unavailable use the plain JSON URL or `/openapi.json` instead.

In terminal B, also in the class directory:

```sh
uv run --locked python -m report_app --local
```

Expect count 2, total 6, mean 3. Stop terminal A with **Ctrl+C** after practice. If the port is already occupied, stop your earlier teaching server or choose another port and pass its URL explicitly to `fetch_observations`. A connection error usually means no server is listening at that address; it is not evidence of invalid records.

**Recovery:** if server setup is incomplete, use the saved response in this notebook and label the result offline. No API account, AI key, or downloaded model is needed.

### 🧩 Exercise 04.1.E1 — Change one parameter and trace the exchange

> **In class · 10 minutes · 20–30**
>
> **Goal:** Connect an encoded request with a useful field in its JSON response.
>
> **Start here:** The prepared request above, saved_response("north.json"), and the supplied service.
>
> **Task:** Prepare GET /observations with label=north; show the URL, inspect status, decode the saved body, and extract value. Name the frontend, backend, and data source. If the local server is ready, repeat with a real request in the terminal and record that separately.
>
> **Check:** The prepared URL ends in ?label=north and the value is 2.5. Your notes distinguish a prepared request, a saved response, a direct Python call, and a live HTTP exchange.
>
> **Hint:** Use params={"label": "north"}; prepare() sends nothing. Index the decoded list before reading its dictionary field.

In [ ]:
# Your attempt for 04.1.E1.
# Record code, observations, and your explanation here.

In [ ]:
# Reveal only after attempting the exercise.
# Remove the leading '# ' from the next line and run this cell.
# %load solutions/exercise_04_1_e1.py

## Guided follow-up — Independent work · 15 minutes

Spend five minutes predicting the response for no label, `north`, and `unknown`. Spend five checking the supplied route directly or trying the explicit local server. Spend five explaining why an empty list can be valid but a connection failure cannot be represented by an empty list. Include your completion check in these minutes.

**Completion:** draw the exchange and identify which program runs each line. Explain why browser and Requests clients can consume the same endpoint. No HTML, JavaScript, database, or deployment work is required.

## References

- [FastAPI first steps](https://fastapi.tiangolo.com/tutorial/first-steps/)
- [FastAPI query parameters](https://fastapi.tiangolo.com/tutorial/query-params/)
- [Requests quickstart](https://requests.readthedocs.io/en/latest/user/quickstart/)
- [Instructor agenda and verification](INSTRUCTOR.md)